In [ ]:
import argparse
import re
import itertools
import pandas as pd
import hydra
from hydra.core.global_hydra import GlobalHydra
import torch

# Import run to trigger OmegaConf resolvers
import topobench.run as tb_run 
from topobench.data.preprocessor import PreProcessor
from topobench.dataloader import TBDataloader

def count_detailed_parameters(model: torch.nn.Module, only_trainable: bool = True):
    """Counts parameters and groups them by their top-level module name."""
    total_params = 0
    breakdown = {"Backbone": 0, "Feature_Encoder": 0, "Readout": 0, "Other": 0}

    for name, p in model.named_parameters():
        if only_trainable and not p.requires_grad:
            continue
        
        num = p.numel()
        total_params += num
        
        # Check the name to categorize it. 
        # We check for "backbone" but exclude "wrapper" just in case you have a backbone_wrapper!
        if "backbone" in name and "wrapper" not in name:
            breakdown["Backbone"] += num
        elif "feature_encoder" in name:
            breakdown["Feature_Encoder"] += num
        elif "readout" in name:
            breakdown["Readout"] += num
        else:
            breakdown["Other"] += num
            
    return total_params, breakdown

def parse_bash_array(filepath: str, array_name: str) -> list[str]:
    """Extracts elements from a bash array in a given file."""
    with open(filepath, 'r') as f:
        content = f.read()
    
    # Match the array definition block
    pattern = rf"{array_name}=\((.*?)\)"
    match = re.search(pattern, content, re.DOTALL)
    
    if not match:
        raise ValueError(f"Could not find array '{array_name}' in {filepath}")
    
    # Split by whitespace and strip quotes
    items = match.group(1).split()
    return [item.strip('"\'') for item in items]

def get_hydra_val(val: str) -> str:
    """Extracts the hydra value from the 'alias::hydra_value' syntax."""
    return val.split('::', 1)[1] if '::' in val else val

def main(sh_path):
    print(f"Parsing parameters from {sh_path}...")
    
    # 2. Extract arrays
    models_raw = parse_bash_array(sh_path, "models")
    datasets_raw = parse_bash_array(sh_path, "datasets")
    neighborhoods_raw = parse_bash_array(sh_path, "neighborhoods")
    encodings_raw = parse_bash_array(sh_path, "encodings")
    num_layers_raw = parse_bash_array(sh_path, "num_layers")
    hidden_channels_raw = parse_bash_array(sh_path, "hidden_channels")

    # 3. Process values
    models = [get_hydra_val(m) for m in models_raw]
    # FIX the dataset to the first one only
    dataset = get_hydra_val(datasets_raw[0]) 
    neighborhoods = [get_hydra_val(n) for n in neighborhoods_raw]
    encodings = [get_hydra_val(e) for e in encodings_raw]
    num_layers = num_layers_raw
    hidden_channels = hidden_channels_raw

    print(f"Fixed Dataset: {dataset}")
    
    # 4. Generate Combinations
    combinations = list(itertools.product(
        models, neighborhoods, encodings, num_layers, hidden_channels
    ))
    
    results = []
    print(f"Total combinations to evaluate: {len(combinations)}")
    print("-" * 60)

    # 5. Initialize Hydra API
    GlobalHydra.instance().clear()
    # Adjust config_path to point to your configs directory relative to this script's execution context
    hydra.initialize(version_base="1.3", config_path="../configs") 

    for idx, (model, nhood, enc, layers, channels) in enumerate(combinations):
        
        # Filter invalid combos
        if model.startswith('cell/') and dataset.startswith('simplicial/'):
            print(f"[{idx+1}/{len(combinations)}] Skipping invalid combo (Cell model + Simplicial dataset)")
            continue

        # Build Hydra Overrides
        overrides = [
            f"model={model}",
            f"dataset={dataset}",
            f"model.preprocessing_params.neighborhoods={nhood}",
            f"model.preprocessing_params.encodings={enc}",
            f"model.backbone.n_layers={layers}",
            f"model.feature_encoder.out_channels={channels}",
            "train=False", 
            "test=False",
            "trainer.accelerator=cpu",
            "trainer.devices=auto"
        ]

        try:
            # Compose the config
            cfg = hydra.compose(config_name="run.yaml", overrides=overrides)
            
            # --- DATALOADER INSTANTIATION (Crucial for dynamic shape inference) ---
            dataset_loader = hydra.utils.instantiate(cfg.dataset.loader)
            loaded_dataset, dataset_dir = dataset_loader.load()
            
            transform_config = (
                hydra.utils.instantiate(cfg.transforms)
                if cfg.get("transforms", None) is not None
                else None
            )
            
            preprocessor = PreProcessor(loaded_dataset, dataset_dir, transform_config)
            dataset_train, dataset_val, dataset_test = preprocessor.load_dataset_splits(cfg.dataset.split_params)
            
            datamodule = TBDataloader(
                dataset_train=dataset_train,
                dataset_val=dataset_val,
                dataset_test=dataset_test,
                **cfg.dataset.get("dataloader_params", {}),
            )
            # ----------------------------------------------------------------------
            
            # Instantiate the model now that dataloader/preprocessor has run
            instantiated_model = hydra.utils.instantiate(
                cfg.model,
                evaluator=cfg.evaluator,
                optimizer=cfg.optimizer,
                loss=cfg.loss,
            )
            
            # Count params
            total_params, breakdown = count_detailed_parameters(instantiated_model)
            
            results.append({
                "Model": model,
                "Neighborhood": nhood,
                "Encoding": enc,
                "Layers": layers,
                "Channels": channels,
                "Total_Params": total_params,
                "Backbone_Params": breakdown["Backbone"],
                "Encoder_Params": breakdown["Feature_Encoder"],
                "Readout_Params": breakdown["Readout"],
                "Other_Params": breakdown["Other"]
            })
            
            print(f"[{idx+1}/{len(combinations)}] SUCCESS | Total: {total_params:,} (Backbone: {breakdown['Backbone']:,}) | L:{layers} C:{channels} M:{model}")

        except Exception as e:
            print(f"[{idx+1}/{len(combinations)}] FAILED  | L:{layers} C:{channels} M:{model}")
            print(f"   Error: {e}")

    # 6. Export Results
    print("-" * 60)
    df = pd.DataFrame(results)
    
    if not df.empty:
        df = df.sort_values(by=["Model", "Backbone_Params"])
        out_file = "parameter_sweep_results.csv"
        df.to_csv(out_file, index=False)
        print(f"\nSaved results to {out_file}")
    else:
        print("No successful combinations to report.")

if __name__ == "__main__":
    sh_path = "../scripts/hopse_m.sh"
    main(sh_path)

Parsing parameters from ../scripts/hopse_m.sh...
Fixed Dataset: graph/MUTAG
Total combinations to evaluate: 120
------------------------------------------------------------
Transform parameters are the same, using existing data_dir: /home/marco/Documents/phd/TopoBench/datasets/graph/TUDataset/MUTAG/graph2simplicial_lifting_hopse_encoding/2357549870


/home/marco/Documents/phd/TopoBench/topobench/nn/encoders/hopse_encoder.py:77: UserWarning: Argument `batch_norm` is deprecated, please use `norm` to specify normalization layer.
  MLP(


[1/120] SUCCESS | Total: 653,186 (Backbone: 252,288) | L:1 C:128 M:simplicial/hopse_m
Transform parameters are the same, using existing data_dir: /home/marco/Documents/phd/TopoBench/datasets/graph/TUDataset/MUTAG/graph2simplicial_lifting_hopse_encoding/2357549870
[2/120] SUCCESS | Total: 2,551,554 (Backbone: 996,096) | L:1 C:256 M:simplicial/hopse_m
Transform parameters are the same, using existing data_dir: /home/marco/Documents/phd/TopoBench/datasets/graph/TUDataset/MUTAG/graph2simplicial_lifting_hopse_encoding/2357549870
[3/120] SUCCESS | Total: 904,706 (Backbone: 503,808) | L:2 C:128 M:simplicial/hopse_m
Transform parameters are the same, using existing data_dir: /home/marco/Documents/phd/TopoBench/datasets/graph/TUDataset/MUTAG/graph2simplicial_lifting_hopse_encoding/2357549870
[4/120] SUCCESS | Total: 3,546,114 (Backbone: 1,990,656) | L:2 C:256 M:simplicial/hopse_m
Transform parameters are the same, using existing data_dir: /home/marco/Documents/phd/TopoBench/datasets/graph/TUDat

KeyError: 'Params'